In [7]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, f_classif, f_regression

In [14]:
path = "../../notebooks/06_cb_regression/dfs_taxon_p/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded df_1 with shape (1485, 109)
Loaded df_10 with shape (3926, 148)
Loaded df_11 with shape (1289, 163)
Loaded df_12 with shape (4677, 176)
Loaded df_13 with shape (1003, 172)
Loaded df_14 with shape (7742, 161)
Loaded df_15 with shape (1223, 160)
Loaded df_16 with shape (379, 113)
Loaded df_17 with shape (803, 151)
Loaded df_18 with shape (1164, 152)
Loaded df_19 with shape (500, 134)
Loaded df_2 with shape (352, 97)
Loaded df_20 with shape (508, 209)
Loaded df_21 with shape (2663, 184)
Loaded df_22 with shape (152, 175)
Loaded df_3 with shape (4996, 141)
Loaded df_4 with shape (596, 141)
Loaded df_5 with shape (2313, 131)
Loaded df_6 with shape (2134, 142)
Loaded df_7 with shape (524, 107)
Loaded df_8 with shape (548, 123)
Loaded df_9 with shape (10254, 169)


In [15]:
from types import SimpleNamespace

# Después de llenar `dfs`:
d = SimpleNamespace(**dfs)

# Usos:
d.df_1.head()
d.df_10.shape

(3926, 148)

In [17]:
cleandf = d.df_1

In [18]:
def crear_modelo(cleandf):
    cleandf = cleandf[cleandf['IBD'].notna()]
    X = cleandf.drop(columns=['IBD','IBD_EQR','IBD_EQR_Status'])
    y = cleandf['IBD']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    # 2) Identify column types
    num_cols = X.select_dtypes(include=['number']).columns
    cat_cols = X.columns.difference(num_cols)
    # 3) Preprocess
    # this preprocessor can handle missing values   
    pre = ColumnTransformer([
        # numerical features
        ('num', Pipeline([
            # imputation and scaling
            ('imp', SimpleImputer(strategy='median')),
            # scaling (RF doesn't need it, but other models might)
            # ('scaler', StandardScaler(with_mean=False))  # stays sparse with OHE
        ]), num_cols),
        # categorical features
        ('cat', Pipeline([
            # imputation as None being another category
            ('imp', SimpleImputer(strategy='constant', fill_value='None')),
            # one-hot encoding, ignoring unknown categories during inference
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)

    ])
    # 4) Model
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, random_state=42, ))  # for regression
    ])  
    clf.fit(X_tr, y_tr)
    print("R2 train:", clf.score(X_tr, y_tr))
    print("R2 valid:", clf.score(X_te, y_te))

    return clf

In [19]:
clf = crear_modelo(cleandf)

R2 train: 0.9571326307216331
R2 valid: 0.7015659552614844
